# Thai STISR: dataset generation

Generates the synthetic HR / LR / 3-zone mask dataset on Colab (Linux), where
Pillow has Raqm, so Thai tone marks and vowels are positioned correctly.

## 1. Clone the repo

In [ ]:
![ -d thai-stisr ] || git clone https://github.com/manatsanancha-sys/thai-stisr.git
%cd thai-stisr

## 2. Install requirements

In [ ]:
!pip install -q -r requirements.txt

## 3. Download fonts (Sarabun, Noto Sans Thai, OFL)

In [ ]:
%%bash
mkdir -p fonts
B=https://raw.githubusercontent.com/google/fonts/main/ofl
curl -fsSL -o fonts/Sarabun-Regular.ttf $B/sarabun/Sarabun-Regular.ttf
curl -fsSL -o fonts/Sarabun-Medium.ttf  $B/sarabun/Sarabun-Medium.ttf
curl -fsSL -o fonts/NotoSansThai.ttf   "$B/notosansthai/NotoSansThai%5Bwdth,wght%5D.ttf"
ls -l fonts

## 4. Check Raqm (needed for correct Thai mark placement)

In [ ]:
from PIL import features
print("raqm:", features.check("raqm"), features.version("raqm"))
assert features.check("raqm"), "Pillow has no Raqm: Thai marks will render wrong"

## 5. Generate the dataset

In [ ]:
!python make_dataset.py --n 3000 --scale 4 --out data

## 6. Preview: HR | LR (nearest) | bicubic | mask (orange = upper, grey = middle, blue = lower)

In [ ]:
!python preview.py data 8 train
from IPython.display import Image, display
display(Image("preview.png"))

## 7. Hard words

Same rendering pipeline, one column pair (HR, mask overlay) per font.
Check that tone marks do not overlap upper vowels or tall letters (ป ฝ ฟ).

In [ ]:
import numpy as np
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display
import make_dataset as md

HARD = ["ที่", "ปี่", "พี่", "ฟื้น", "ฝั่ง", "ป่า", "น้ำ"]
COLS = [(230, 90, 20), (60, 60, 60), (30, 90, 230)]  # upper, middle, lower
fonts = md.load_fonts("fonts")
label_font = ImageFont.truetype("fonts/Sarabun-Regular.ttf", 28)

def overlay(mask):
    ov = np.full(mask.shape, 255.0)
    for k, col in enumerate(COLS):
        a = mask[..., k:k + 1] / 255.0
        ov = ov * (1 - a) + np.array(col) * a
    return ov.astype(np.uint8)

rows = []
header = PILImage.new("RGB", (100 + 512 * len(fonts), 30), (245, 245, 245))
for i, (name, _, _) in enumerate(fonts):
    ImageDraw.Draw(header).text((100 + 512 * i + 4, 6), name, fill=(0, 0, 0))
rows.append(np.asarray(header))
for w in HARD:
    label = PILImage.new("RGB", (100, 64), (255, 255, 255))
    ImageDraw.Draw(label).text((8, 8), w, font=label_font, fill=(0, 0, 0))
    tiles = [np.asarray(label)]
    for _, _, f in fonts:
        hr, lr, mask, _ = md.make_sample(w, f, 64, 256, 4, np.random.default_rng(0))
        tiles += [np.stack([hr] * 3, -1), overlay(mask)]
    rows.append(np.concatenate(tiles, 1))
sheet = PILImage.fromarray(np.concatenate(rows, 0))
sheet.save("hard_words.png")
display(sheet)